In [0]:
import gc
import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import re
import time
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import Row, functions as F, types as T


dbutils.widgets.dropdown("MODE", "PLAN", ["PLAN", "BOOTSTRAP", "CANARY", "APPLY"])
dbutils.widgets.text("RUN_ID", "")
dbutils.widgets.text("STATE_SCHEMA", "4_prod.tmp")
dbutils.widgets.text("BRONZE_TABLE", "4_prod.bronze.mill_blob_text")
dbutils.widgets.text("HISTORY_TABLE", "4_prod.logs.mill_blob_extractor_history")
dbutils.widgets.text("MANIFEST_TABLE", "4_prod.6_mgmt.file_manifest")
dbutils.widgets.text("MAX_DOCUMENTS", "100")
dbutils.widgets.text("MAX_ATTEMPTS", "3")
dbutils.widgets.text("LEASE_MINUTES", "120")
dbutils.widgets.text("OCR_MAX_PAGES", "10")
dbutils.widgets.text("OCR_DPI", "170")
dbutils.widgets.text("OCR_PAGE_BATCH_SIZE", "4")
dbutils.widgets.text("OCR_TOTAL_TIMEOUT_SEC", "240")
dbutils.widgets.text("OCR_MIN_CONFIDENCE", "0.35")
dbutils.widgets.dropdown("PREPROCESSING", "none", ["none", "clahe"])
dbutils.widgets.text("OCR_LANGUAGE", "en")
dbutils.widgets.dropdown("OCR_MODEL_VERSION", "PP-OCRv6", ["PP-OCRv6", "PP-OCRv5"])
dbutils.widgets.text("OCR_DEVICE", "gpu:0")
dbutils.widgets.text(
    "MODEL_CACHE_VOLUME",
    "/Volumes/6_mgmt/default/scripts/paddleocr_models/ppocrv6_en",
)
dbutils.widgets.dropdown("ALLOW_MODEL_DOWNLOAD", "false", ["false", "true"])
dbutils.widgets.text("CANARY_VOLUME_PATH", "")


MODE = dbutils.widgets.get("MODE").strip().upper()
RUN_ID = dbutils.widgets.get("RUN_ID").strip()
STATE_SCHEMA = dbutils.widgets.get("STATE_SCHEMA").strip()
BRONZE = dbutils.widgets.get("BRONZE_TABLE").strip()
HISTORY = dbutils.widgets.get("HISTORY_TABLE").strip()
MANIFEST = dbutils.widgets.get("MANIFEST_TABLE").strip()
MAX_DOCUMENTS = int(dbutils.widgets.get("MAX_DOCUMENTS") or "100")
MAX_ATTEMPTS = int(dbutils.widgets.get("MAX_ATTEMPTS") or "3")
LEASE_MINUTES = int(dbutils.widgets.get("LEASE_MINUTES") or "120")
OCR_MAX_PAGES = int(dbutils.widgets.get("OCR_MAX_PAGES") or "10")
OCR_DPI = int(dbutils.widgets.get("OCR_DPI") or "170")
OCR_PAGE_BATCH_SIZE = int(dbutils.widgets.get("OCR_PAGE_BATCH_SIZE") or "4")
OCR_TOTAL_TIMEOUT_SEC = int(dbutils.widgets.get("OCR_TOTAL_TIMEOUT_SEC") or "240")
OCR_MIN_CONFIDENCE = float(dbutils.widgets.get("OCR_MIN_CONFIDENCE") or "0.35")
PREPROCESSING = dbutils.widgets.get("PREPROCESSING").strip().lower()
OCR_LANGUAGE = dbutils.widgets.get("OCR_LANGUAGE").strip()
OCR_MODEL_VERSION = dbutils.widgets.get("OCR_MODEL_VERSION").strip()
OCR_DEVICE = dbutils.widgets.get("OCR_DEVICE").strip()
MODEL_CACHE_VOLUME = dbutils.widgets.get("MODEL_CACHE_VOLUME").strip().rstrip("/")
ALLOW_MODEL_DOWNLOAD = dbutils.widgets.get("ALLOW_MODEL_DOWNLOAD").lower() == "true"
CANARY_VOLUME_PATH = dbutils.widgets.get("CANARY_VOLUME_PATH").strip()

FILE_QUEUE = f"{STATE_SCHEMA}.file_queue"
OCR_QUEUE = f"{STATE_SCHEMA}.blob_ocr_queue"
OCR_OUTPUT = f"{STATE_SCHEMA}.blob_ocr_output"
OCR_PAGE_OUTPUT = f"{STATE_SCHEMA}.blob_ocr_page_output"
OCR_CANARY_OUTPUT = f"{STATE_SCHEMA}.blob_ocr_canary_output"

OCR_PARSER_VERSION = 5
OCR_POST_PROCESSOR_VERSION = 5
MAX_TEXT_CHARS = 10_000_000
SUPPORTED_SOURCE_STATUSES = ("OCR queued", "PDF extraction failed")

if MODE not in {"PLAN", "BOOTSTRAP", "CANARY", "APPLY"}:
    raise ValueError(f"Unsupported MODE={MODE!r}")
if not 1 <= MAX_DOCUMENTS <= 5000:
    raise ValueError("MAX_DOCUMENTS must be between 1 and 5000")
if not 1 <= MAX_ATTEMPTS <= 20:
    raise ValueError("MAX_ATTEMPTS must be between 1 and 20")
if not 1 <= OCR_MAX_PAGES <= 100:
    raise ValueError("OCR_MAX_PAGES must be between 1 and 100")
if not 72 <= OCR_DPI <= 300:
    raise ValueError("OCR_DPI must be between 72 and 300")
if not 1 <= OCR_PAGE_BATCH_SIZE <= 32:
    raise ValueError("OCR_PAGE_BATCH_SIZE must be between 1 and 32")
if not 0.0 <= OCR_MIN_CONFIDENCE <= 1.0:
    raise ValueError("OCR_MIN_CONFIDENCE must be between 0 and 1")
if PREPROCESSING not in {"none", "clahe"}:
    raise ValueError("PREPROCESSING must be none or clahe")


In [0]:
_TABLE_RE = re.compile(r"^[A-Za-z0-9_]+(?:\.[A-Za-z0-9_]+){1,2}$")


def validate_table_name(value):
    if not _TABLE_RE.fullmatch(value or ""):
        raise ValueError(f"Unsafe table identifier: {value!r}")
    return value


def quote_table(value):
    validate_table_name(value)
    return ".".join(f"`{part}`" for part in value.split("."))


def table_exists(value):
    try:
        spark.table(value).schema
        return True
    except Exception as exc:
        message = str(exc).lower()
        if any(
            marker in message
            for marker in (
                "table_or_view_not_found",
                "table or view not found",
                "cannot be found",
                "no such table",
            )
        ):
            return False
        raise


def ensure_columns(table_name, columns):
    existing = {field.name.lower() for field in spark.table(table_name).schema.fields}
    missing = [(name, data_type) for name, data_type in columns if name.lower() not in existing]
    if missing:
        rendered = ", ".join(f"`{name}` {data_type}" for name, data_type in missing)
        spark.sql(f"ALTER TABLE {quote_table(table_name)} ADD COLUMNS ({rendered})")


for _table in (STATE_SCHEMA, BRONZE, HISTORY, MANIFEST, FILE_QUEUE, OCR_QUEUE, OCR_OUTPUT,
               OCR_PAGE_OUTPUT, OCR_CANARY_OUTPUT):
    validate_table_name(_table)


def bootstrap_tables():
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quote_table(OCR_QUEUE)} (
      run_id STRING,
      EVENT_ID BIGINT,
      ADC_UPDT TIMESTAMP,
      raw_sha256 STRING,
      volume_path STRING,
      status STRING,
      attempt_count INT,
      lease_owner STRING,
      lease_expires_ts TIMESTAMP,
      last_error STRING,
      created_ts TIMESTAMP,
      updated_ts TIMESTAMP,
      completed_ts TIMESTAMP
    ) USING DELTA
    """)
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quote_table(OCR_OUTPUT)} (
      run_id STRING,
      EVENT_ID BIGINT,
      ADC_UPDT TIMESTAMP,
      raw_sha256 STRING,
      volume_path STRING,
      text STRING,
      text_length BIGINT,
      status STRING,
      parse_status STRING,
      pages_processed INT,
      page_count INT,
      attempt_count INT,
      engine STRING,
      model_version STRING,
      paddleocr_version STRING,
      paddle_version STRING,
      device STRING,
      line_count INT,
      mean_confidence DOUBLE,
      min_confidence DOUBLE,
      truncated BOOLEAN,
      metrics STRING,
      output_ts TIMESTAMP
    ) USING DELTA
    """)
    ensure_columns(OCR_OUTPUT, [
        ("engine", "STRING"),
        ("model_version", "STRING"),
        ("paddleocr_version", "STRING"),
        ("paddle_version", "STRING"),
        ("device", "STRING"),
        ("line_count", "INT"),
        ("mean_confidence", "DOUBLE"),
        ("min_confidence", "DOUBLE"),
        ("truncated", "BOOLEAN"),
    ])
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quote_table(OCR_PAGE_OUTPUT)} (
      run_id STRING,
      EVENT_ID BIGINT,
      ADC_UPDT TIMESTAMP,
      raw_sha256 STRING,
      page_index INT,
      text STRING,
      line_count INT,
      mean_confidence DOUBLE,
      min_confidence DOUBLE,
      lines_json STRING,
      status STRING,
      elapsed_seconds DOUBLE,
      engine STRING,
      model_version STRING,
      output_ts TIMESTAMP
    ) USING DELTA
    """)
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {quote_table(OCR_CANARY_OUTPUT)} (
      canary_id STRING,
      run_id STRING,
      EVENT_ID BIGINT,
      ADC_UPDT TIMESTAMP,
      raw_sha256 STRING,
      volume_path STRING,
      text STRING,
      text_length BIGINT,
      status STRING,
      parse_status STRING,
      pages_processed INT,
      page_count INT,
      attempt_count INT,
      engine STRING,
      model_version STRING,
      paddleocr_version STRING,
      paddle_version STRING,
      device STRING,
      line_count INT,
      mean_confidence DOUBLE,
      min_confidence DOUBLE,
      truncated BOOLEAN,
      metrics STRING,
      output_ts TIMESTAMP
    ) USING DELTA
    """)


In [0]:
def package_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None


def configure_paddle_environment(require_staged_cache):
    if not MODEL_CACHE_VOLUME:
        raise ValueError("MODEL_CACHE_VOLUME cannot be empty")
    manifest_path = os.path.join(MODEL_CACHE_VOLUME, "optimum_manifest.json")
    if require_staged_cache and not os.path.exists(manifest_path) and not ALLOW_MODEL_DOWNLOAD:
        raise RuntimeError(
            "PaddleOCR model cache is not staged. Run this notebook once with "
            "MODE=BOOTSTRAP on the GPU job compute, or temporarily set "
            "ALLOW_MODEL_DOWNLOAD=true for a controlled canary. Missing: " + manifest_path
        )
    os.makedirs("/local_disk0/paddleocr_tmp", exist_ok=True)
    if MODE == "BOOTSTRAP" or ALLOW_MODEL_DOWNLOAD:
        os.makedirs(MODEL_CACHE_VOLUME, exist_ok=True)
    os.environ["PADDLEX_HOME"] = MODEL_CACHE_VOLUME
    os.environ["TMPDIR"] = "/local_disk0/paddleocr_tmp"
    os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
    os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "False"
    os.environ.setdefault("FLAGS_fraction_of_gpu_memory_to_use", "0.85")
    return manifest_path


def gpu_report():
    report = {
        "paddleocr": package_version("paddleocr"),
        "paddlepaddle_gpu": package_version("paddlepaddle-gpu"),
        "paddlepaddle": package_version("paddlepaddle"),
        "PyMuPDF": package_version("PyMuPDF"),
        "model_cache_volume": MODEL_CACHE_VOLUME,
        "device_requested": OCR_DEVICE,
    }
    try:
        import paddle

        report["paddle_module_version"] = getattr(paddle, "__version__", None)
        report["compiled_with_cuda"] = bool(paddle.device.is_compiled_with_cuda())
        report["gpu_count"] = int(paddle.device.cuda.device_count())
        if report["gpu_count"]:
            try:
                report["gpu_name"] = paddle.device.cuda.get_device_name(0)
            except Exception:
                report["gpu_name"] = None
    except Exception as exc:
        report["paddle_import_error"] = f"{type(exc).__name__}: {str(exc)[:1000]}"
    return report


def build_engine():
    from paddleocr import PaddleOCR

    modern_kwargs = {
        "lang": OCR_LANGUAGE,
        "ocr_version": OCR_MODEL_VERSION,
        "device": OCR_DEVICE,
        "use_doc_orientation_classify": False,
        "use_doc_unwarping": False,
        "use_textline_orientation": False,
        "text_recognition_batch_size": max(1, OCR_PAGE_BATCH_SIZE * 2),
        "text_rec_score_thresh": OCR_MIN_CONFIDENCE,
    }
    try:
        return PaddleOCR(**modern_kwargs), "paddleocr_v3"
    except TypeError as exc:
        message = str(exc).lower()
        if not any(marker in message for marker in ("unexpected keyword", "unknown argument")):
            raise
        legacy_kwargs = {
            "lang": OCR_LANGUAGE,
            "use_gpu": OCR_DEVICE.startswith("gpu"),
            "use_angle_cls": False,
            "show_log": False,
            "det_limit_side_len": 1600,
        }
        return PaddleOCR(**legacy_kwargs), "paddleocr_v2_compat"


def extract_result_payload(result):
    payload = getattr(result, "json", None)
    if callable(payload):
        payload = payload()
    if payload is None and isinstance(result, dict):
        payload = result
    if isinstance(payload, str):
        payload = json.loads(payload)
    if isinstance(payload, dict) and isinstance(payload.get("res"), dict):
        payload = payload["res"]
    if not isinstance(payload, dict):
        raise TypeError(f"Unsupported PaddleOCR result type: {type(result).__name__}")
    return payload


def normalize_polygon(value):
    if value is None:
        return None
    if hasattr(value, "tolist"):
        value = value.tolist()
    return [[int(round(float(point[0]))), int(round(float(point[1])))] for point in value]


def parse_modern_result(result):
    payload = extract_result_payload(result)
    texts = list(payload.get("rec_texts") or [])
    scores_raw = payload.get("rec_scores")
    polygons_raw = payload.get("rec_polys")
    boxes_raw = payload.get("rec_boxes")
    scores = scores_raw.tolist() if hasattr(scores_raw, "tolist") else list(scores_raw or [])
    polygons = polygons_raw.tolist() if hasattr(polygons_raw, "tolist") else list(polygons_raw or [])
    boxes = boxes_raw.tolist() if hasattr(boxes_raw, "tolist") else list(boxes_raw or [])
    lines = []
    for index, text_value in enumerate(texts):
        text_value = str(text_value or "").strip()
        score = float(scores[index]) if index < len(scores) and scores[index] is not None else None
        if not text_value or (score is not None and score < OCR_MIN_CONFIDENCE):
            continue
        polygon = normalize_polygon(polygons[index]) if index < len(polygons) else None
        box = boxes[index] if index < len(boxes) else None
        if box is not None and hasattr(box, "tolist"):
            box = box.tolist()
        if box is None and polygon:
            xs = [point[0] for point in polygon]
            ys = [point[1] for point in polygon]
            box = [min(xs), min(ys), max(xs), max(ys)]
        box = [int(round(float(value))) for value in box] if box is not None else None
        lines.append({"text": text_value, "score": score, "polygon": polygon, "box": box})
    lines.sort(key=lambda item: (
        item["box"][1] if item.get("box") else 0,
        item["box"][0] if item.get("box") else 0,
    ))
    return lines


def parse_legacy_result(result):
    detections = result
    if result and len(result) == 1 and isinstance(result[0], list):
        detections = result[0]
    lines = []
    for detection in detections or []:
        if not detection or len(detection) < 2:
            continue
        polygon, recognition = detection[0], detection[1]
        text_value = str(recognition[0] or "").strip()
        score = float(recognition[1]) if len(recognition) > 1 else None
        if not text_value or (score is not None and score < OCR_MIN_CONFIDENCE):
            continue
        polygon = normalize_polygon(polygon)
        xs = [point[0] for point in polygon]
        ys = [point[1] for point in polygon]
        lines.append({
            "text": text_value,
            "score": score,
            "polygon": polygon,
            "box": [min(xs), min(ys), max(xs), max(ys)],
        })
    lines.sort(key=lambda item: (item["box"][1], item["box"][0]))
    return lines


def predict_images(engine, engine_kind, images):
    if engine_kind == "paddleocr_v3":
        return [parse_modern_result(value) for value in engine.predict(images)]
    return [parse_legacy_result(engine.ocr(image, cls=False)) for image in images]


In [0]:
def candidate_frame():
    bronze_candidates = (
        spark.table(BRONZE)
        .filter(F.col("CONTENT_TYPE") == "application/pdf")
        .filter(F.col("STATUS").isin(*SUPPORTED_SOURCE_STATUSES))
        .filter(F.col("raw_sha256").isNotNull())
        .filter(F.col("BLOB_TEXT").isNull() | (F.length(F.trim(F.col("BLOB_TEXT"))) == 0))
        .select("EVENT_ID", "ADC_UPDT", "raw_sha256")
    )

    queue_candidates = (
        spark.table(FILE_QUEUE).alias("q")
        .filter(
            (F.col("q.status") == "extracted")
            & (F.col("q.content_type") == "application/pdf")
            & F.col("q.volume_path").isNotNull()
        )
        .join(
            bronze_candidates.alias("b"),
            (F.col("q.EVENT_ID") == F.col("b.EVENT_ID"))
            & F.col("q.ADC_UPDT").eqNullSafe(F.col("b.ADC_UPDT"))
            & (F.col("q.raw_sha256") == F.col("b.raw_sha256")),
            "inner",
        )
        .select(
            F.coalesce(F.col("q.run_id"), F.lit("legacy")).alias("run_id"),
            F.col("q.EVENT_ID").cast("long").alias("EVENT_ID"),
            F.col("q.ADC_UPDT").alias("ADC_UPDT"),
            F.col("q.raw_sha256").alias("raw_sha256"),
            F.col("q.volume_path").alias("volume_path"),
        )
    )

    manifest_candidates = (
        spark.table(MANIFEST).alias("m")
        .filter(
            (F.col("m.STATUS") == "Extracted")
            & (F.col("m.content_type") == "application/pdf")
            & F.col("m.volume_path").isNotNull()
        )
        .join(
            bronze_candidates.alias("b"),
            (F.col("m.EVENT_ID") == F.col("b.EVENT_ID"))
            & F.col("m.ADC_UPDT").eqNullSafe(F.col("b.ADC_UPDT"))
            & (F.col("m.raw_sha256") == F.col("b.raw_sha256")),
            "inner",
        )
        .select(
            F.coalesce(F.col("m.run_id"), F.lit("legacy")).alias("run_id"),
            F.col("m.EVENT_ID").cast("long").alias("EVENT_ID"),
            F.col("m.ADC_UPDT").alias("ADC_UPDT"),
            F.col("m.raw_sha256").alias("raw_sha256"),
            F.col("m.volume_path").alias("volume_path"),
        )
    )

    return (
        queue_candidates.unionByName(manifest_candidates)
        .dropDuplicates(["EVENT_ID", "ADC_UPDT", "raw_sha256"])
        .withColumn("status", F.lit("pending"))
        .withColumn("attempt_count", F.lit(0).cast("int"))
        .withColumn("lease_owner", F.lit(None).cast("string"))
        .withColumn("lease_expires_ts", F.lit(None).cast("timestamp"))
        .withColumn("last_error", F.lit(None).cast("string"))
        .withColumn("created_ts", F.current_timestamp())
        .withColumn("updated_ts", F.current_timestamp())
        .withColumn("completed_ts", F.lit(None).cast("timestamp"))
    )


def manual_canary_frame(path):
    if not path:
        return None
    if not os.path.exists(path):
        raise FileNotFoundError(f"CANARY_VOLUME_PATH does not exist: {path}")
    with open(path, "rb") as handle:
        digest = hashlib.sha256(handle.read()).hexdigest()
    event_id = -int(digest[:12], 16)
    schema = T.StructType([
        T.StructField("run_id", T.StringType(), False),
        T.StructField("EVENT_ID", T.LongType(), False),
        T.StructField("ADC_UPDT", T.TimestampType(), True),
        T.StructField("raw_sha256", T.StringType(), False),
        T.StructField("volume_path", T.StringType(), False),
        T.StructField("attempt_count", T.IntegerType(), False),
    ])
    return spark.createDataFrame(
        [("manual_canary", event_id, None, digest, path, 0)],
        schema=schema,
    )


required_sources = (BRONZE, HISTORY, MANIFEST, FILE_QUEUE)
missing_sources = [table_name for table_name in required_sources if not table_exists(table_name)]
if missing_sources:
    raise RuntimeError(f"Missing required source tables: {missing_sources}")

if MODE == "BOOTSTRAP":
    bootstrap_tables()
elif MODE in {"CANARY", "APPLY"}:
    missing_control = [
        table_name for table_name in (OCR_QUEUE, OCR_OUTPUT, OCR_PAGE_OUTPUT, OCR_CANARY_OUTPUT)
        if not table_exists(table_name)
    ]
    if missing_control:
        raise RuntimeError(
            f"Missing OCR control tables: {missing_control}. Run MODE=BOOTSTRAP once."
        )

candidates = candidate_frame().persist()
candidate_count = int(candidates.count())
queue_summary = []
if table_exists(OCR_QUEUE):
    queue_summary = [row.asDict() for row in (
        spark.table(OCR_QUEUE)
        .groupBy("status")
        .count()
        .orderBy(F.desc("count"))
        .collect()
    )]

print(json.dumps({
    "mode": MODE,
    "run_id": RUN_ID or None,
    "new_candidates": candidate_count,
    "queue": queue_summary,
}, indent=2, default=str))

if MODE == "PLAN":
    display(candidates.select("run_id", "EVENT_ID", "ADC_UPDT", "raw_sha256", "volume_path").limit(100))
    candidates.unpersist()
    dbutils.notebook.exit(json.dumps({
        "status": "plan",
        "new_candidates": candidate_count,
        "queue": queue_summary,
    }, default=str))


In [0]:
manifest_path = configure_paddle_environment(require_staged_cache=MODE != "BOOTSTRAP")

if MODE == "BOOTSTRAP":
    report = gpu_report()
    if not report.get("compiled_with_cuda") or int(report.get("gpu_count") or 0) < 1:
        raise RuntimeError("BOOTSTRAP must run on GPU compute: " + json.dumps(report, default=str))
    engine, engine_kind = build_engine()
    import cv2
    import numpy as np

    image = np.full((320, 1280, 3), 255, dtype=np.uint8)
    cv2.putText(
        image,
        "RDE PADDLE OCR GPU READY 12345",
        (30, 180),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.6,
        (0, 0, 0),
        3,
        cv2.LINE_AA,
    )
    lines = predict_images(engine, engine_kind, [image])[0]
    recognized = " ".join(line["text"] for line in lines)
    report.update({
        "engine_kind": engine_kind,
        "model_version": OCR_MODEL_VERSION,
        "language": OCR_LANGUAGE,
        "synthetic_recognized_text": recognized,
        "staged_at_utc": datetime.now(timezone.utc).isoformat(),
    })
    if not recognized.strip():
        raise RuntimeError("PaddleOCR GPU bootstrap returned no synthetic text: " + json.dumps(report))
    os.makedirs(MODEL_CACHE_VOLUME, exist_ok=True)
    with open(manifest_path, "w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, sort_keys=True)
    del engine
    gc.collect()
    candidates.unpersist()
    print(json.dumps(report, indent=2, default=str))
    dbutils.notebook.exit(json.dumps({"status": "bootstrapped", **report}, default=str))


In [0]:
if MODE == "APPLY" and candidate_count:
    (
        DeltaTable.forName(spark, OCR_QUEUE)
        .alias("t")
        .merge(
            candidates.alias("s"),
            "t.EVENT_ID=s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT "
            "AND t.raw_sha256=s.raw_sha256",
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

if MODE == "CANARY" and CANARY_VOLUME_PATH:
    selected = manual_canary_frame(CANARY_VOLUME_PATH)
else:
    source = spark.table(OCR_QUEUE) if MODE == "APPLY" else candidates
    eligible = source.filter(
        F.col("status").isin("pending", "retryable")
        | (
            (F.col("status") == "processing")
            & (
                F.col("lease_expires_ts").isNull()
                | (F.col("lease_expires_ts") < F.current_timestamp())
            )
        )
    )
    if "attempt_count" in eligible.columns:
        eligible = eligible.filter(F.coalesce(F.col("attempt_count"), F.lit(0)) < MAX_ATTEMPTS)
    selected = (
        eligible
        .withColumn(
            "_run_priority",
            F.when(F.lit(bool(RUN_ID)) & (F.col("run_id") == F.lit(RUN_ID)), F.lit(0))
             .otherwise(F.lit(1)),
        )
        .orderBy("_run_priority", "created_ts", "EVENT_ID")
        .limit(MAX_DOCUMENTS)
        .select("run_id", "EVENT_ID", "ADC_UPDT", "raw_sha256", "volume_path", "attempt_count")
    )

selected_count = int(selected.count())
if selected_count == 0:
    candidates.unpersist()
    result = {"status": "success", "mode": MODE, "selected": 0, "new_candidates": candidate_count}
    dbutils.jobs.taskValues.set(key="OCR_RESULT", value=json.dumps(result))
    dbutils.notebook.exit(json.dumps(result))

lease_owner = uuid.uuid4().hex
if MODE == "APPLY":
    selected.select("EVENT_ID", "ADC_UPDT", "raw_sha256").createOrReplaceTempView(
        "_optimum_paddle_ocr_claims"
    )
    spark.sql(f"""
    MERGE INTO {quote_table(OCR_QUEUE)} t
    USING _optimum_paddle_ocr_claims s
    ON t.EVENT_ID=s.EVENT_ID
     AND t.ADC_UPDT <=> s.ADC_UPDT
     AND t.raw_sha256=s.raw_sha256
    WHEN MATCHED AND (
      t.status IN ('pending','retryable')
      OR t.lease_expires_ts IS NULL
      OR t.lease_expires_ts < current_timestamp()
    ) THEN UPDATE SET
      t.status='processing',
      t.attempt_count=coalesce(t.attempt_count,0)+1,
      t.lease_owner='{lease_owner}',
      t.lease_expires_ts=current_timestamp() + INTERVAL {LEASE_MINUTES} MINUTES,
      t.last_error=NULL,
      t.updated_ts=current_timestamp()
    """)
    selected = (
        spark.table(OCR_QUEUE)
        .filter(F.col("lease_owner") == lease_owner)
        .select("run_id", "EVENT_ID", "ADC_UPDT", "raw_sha256", "volume_path", "attempt_count")
    )
    selected_count = int(selected.count())

candidates.unpersist()


In [0]:
report = gpu_report()
if not report.get("compiled_with_cuda") or int(report.get("gpu_count") or 0) < 1:
    raise RuntimeError("PaddleOCR task is not running on GPU compute: " + json.dumps(report, default=str))

engine_started = time.monotonic()
engine, engine_kind = build_engine()
engine_init_seconds = round(time.monotonic() - engine_started, 3)
report["engine_kind"] = engine_kind
report["engine_init_seconds"] = engine_init_seconds
print(json.dumps(report, indent=2, default=str))


result_schema = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("EVENT_ID", T.LongType()),
    T.StructField("ADC_UPDT", T.TimestampType()),
    T.StructField("raw_sha256", T.StringType()),
    T.StructField("volume_path", T.StringType()),
    T.StructField("text", T.StringType()),
    T.StructField("text_length", T.LongType()),
    T.StructField("status", T.StringType()),
    T.StructField("parse_status", T.StringType()),
    T.StructField("pages_processed", T.IntegerType()),
    T.StructField("page_count", T.IntegerType()),
    T.StructField("attempt_count", T.IntegerType()),
    T.StructField("engine", T.StringType()),
    T.StructField("model_version", T.StringType()),
    T.StructField("paddleocr_version", T.StringType()),
    T.StructField("paddle_version", T.StringType()),
    T.StructField("device", T.StringType()),
    T.StructField("line_count", T.IntegerType()),
    T.StructField("mean_confidence", T.DoubleType()),
    T.StructField("min_confidence", T.DoubleType()),
    T.StructField("truncated", T.BooleanType()),
    T.StructField("metrics", T.StringType()),
])

page_schema = T.StructType([
    T.StructField("run_id", T.StringType()),
    T.StructField("EVENT_ID", T.LongType()),
    T.StructField("ADC_UPDT", T.TimestampType()),
    T.StructField("raw_sha256", T.StringType()),
    T.StructField("page_index", T.IntegerType()),
    T.StructField("text", T.StringType()),
    T.StructField("line_count", T.IntegerType()),
    T.StructField("mean_confidence", T.DoubleType()),
    T.StructField("min_confidence", T.DoubleType()),
    T.StructField("lines_json", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("elapsed_seconds", T.DoubleType()),
    T.StructField("engine", T.StringType()),
    T.StructField("model_version", T.StringType()),
])


def preprocess_image(image):
    if PREPROCESSING == "none":
        return image
    import cv2

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def render_pages(document, page_indexes):
    import fitz
    import numpy as np
    import cv2

    images = []
    rendered = []
    errors = []
    for page_index in page_indexes:
        try:
            page = document.load_page(page_index)
            pixmap = page.get_pixmap(
                matrix=fitz.Matrix(OCR_DPI / 72.0, OCR_DPI / 72.0),
                colorspace=fitz.csRGB,
                alpha=False,
            )
            rgb = np.frombuffer(pixmap.samples, dtype=np.uint8).reshape(
                pixmap.height, pixmap.width, pixmap.n
            )
            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            images.append(preprocess_image(bgr))
            rendered.append(page_index)
        except Exception as exc:
            errors.append(f"page={page_index}:render:{type(exc).__name__}:{str(exc)[:300]}")
    return images, rendered, errors


def confidence_stats(lines):
    scores = [float(line["score"]) for line in lines if line.get("score") is not None]
    if not scores:
        return None, None
    return sum(scores) / len(scores), min(scores)


def process_document(row):
    import fitz
    import paddle

    started = time.monotonic()
    errors = []
    page_outputs = []
    all_parts = []
    all_scores = []
    line_count = 0
    pages_processed = 0
    page_count = 0
    soft_timeout = False
    truncated = False
    parse_status = "ocr_error"
    status = "OCR error (PaddleOCR)"

    try:
        document = fitz.open(row.volume_path)
        try:
            page_count = int(document.page_count)
            page_limit = min(page_count, OCR_MAX_PAGES)
            truncated = page_count > OCR_MAX_PAGES
            for batch_start in range(0, page_limit, OCR_PAGE_BATCH_SIZE):
                if time.monotonic() - started >= OCR_TOTAL_TIMEOUT_SEC:
                    soft_timeout = True
                    truncated = True
                    break
                indexes = list(range(batch_start, min(page_limit, batch_start + OCR_PAGE_BATCH_SIZE)))
                images, rendered_indexes, render_errors = render_pages(document, indexes)
                errors.extend(render_errors)
                if not images:
                    continue
                infer_started = time.monotonic()
                try:
                    page_lines = predict_images(engine, engine_kind, images)
                except Exception as exc:
                    errors.append(f"pages={rendered_indexes}:infer:{type(exc).__name__}:{str(exc)[:500]}")
                    try:
                        paddle.device.cuda.empty_cache()
                    except Exception:
                        pass
                    continue
                infer_elapsed = time.monotonic() - infer_started
                per_page_elapsed = infer_elapsed / max(1, len(rendered_indexes))
                for page_index, lines in zip(rendered_indexes, page_lines):
                    page_text = "\n".join(line["text"] for line in lines).strip()
                    page_mean, page_min = confidence_stats(lines)
                    page_outputs.append({
                        "run_id": row.run_id,
                        "EVENT_ID": int(row.EVENT_ID),
                        "ADC_UPDT": row.ADC_UPDT,
                        "raw_sha256": row.raw_sha256,
                        "page_index": int(page_index),
                        "text": page_text or None,
                        "line_count": len(lines),
                        "mean_confidence": page_mean,
                        "min_confidence": page_min,
                        "lines_json": json.dumps(lines[:2000], ensure_ascii=False, default=str),
                        "status": "ocr_page_text" if page_text else "ocr_page_empty",
                        "elapsed_seconds": round(per_page_elapsed, 3),
                        "engine": engine_kind,
                        "model_version": OCR_MODEL_VERSION,
                    })
                    pages_processed += 1
                    line_count += len(lines)
                    all_scores.extend(
                        float(line["score"]) for line in lines if line.get("score") is not None
                    )
                    if page_text:
                        all_parts.append(page_text)
                del images
                gc.collect()
        finally:
            document.close()

        text_value = "\n\n".join(all_parts).strip() or None
        if text_value and len(text_value) > MAX_TEXT_CHARS:
            text_value = text_value[:MAX_TEXT_CHARS]
            truncated = True
            errors.append(f"text_truncated_at={MAX_TEXT_CHARS}")
        if pages_processed < min(page_count, OCR_MAX_PAGES):
            truncated = True
        if text_value:
            parse_status = "ocr_partial" if truncated or errors else "ocr_complete"
            status = "OCR partial (PaddleOCR)" if parse_status == "ocr_partial" else "OCR complete (PaddleOCR)"
        else:
            parse_status = "ocr_error" if errors and pages_processed == 0 else "ocr_empty"
            status = "OCR error (PaddleOCR)" if parse_status == "ocr_error" else "OCR produced no text (PaddleOCR)"
    except Exception as exc:
        text_value = None
        parse_status = "ocr_error"
        status = "OCR error (PaddleOCR)"
        errors.append(f"document:{type(exc).__name__}:{str(exc)[:1000]}")
        try:
            paddle.device.cuda.empty_cache()
        except Exception:
            pass

    elapsed = time.monotonic() - started
    mean_confidence = sum(all_scores) / len(all_scores) if all_scores else None
    min_confidence = min(all_scores) if all_scores else None
    metrics = {
        "elapsed_seconds": round(elapsed, 3),
        "engine_init_seconds": engine_init_seconds,
        "engine": engine_kind,
        "model_version": OCR_MODEL_VERSION,
        "language": OCR_LANGUAGE,
        "device": OCR_DEVICE,
        "dpi": OCR_DPI,
        "page_batch_size": OCR_PAGE_BATCH_SIZE,
        "min_confidence": OCR_MIN_CONFIDENCE,
        "preprocessing": PREPROCESSING,
        "pages_processed": pages_processed,
        "page_count": page_count,
        "line_count": line_count,
        "soft_timeout": soft_timeout,
        "truncated": truncated,
        "errors": errors[:50],
    }
    result = {
        "run_id": row.run_id,
        "EVENT_ID": int(row.EVENT_ID),
        "ADC_UPDT": row.ADC_UPDT,
        "raw_sha256": row.raw_sha256,
        "volume_path": row.volume_path,
        "text": text_value,
        "text_length": len(text_value) if text_value else None,
        "status": status,
        "parse_status": parse_status,
        "pages_processed": pages_processed,
        "page_count": page_count,
        "attempt_count": int(row.attempt_count or 0),
        "engine": engine_kind,
        "model_version": OCR_MODEL_VERSION,
        "paddleocr_version": report.get("paddleocr"),
        "paddle_version": report.get("paddle_module_version") or report.get("paddlepaddle_gpu"),
        "device": OCR_DEVICE,
        "line_count": line_count,
        "mean_confidence": mean_confidence,
        "min_confidence": min_confidence,
        "truncated": bool(truncated),
        "metrics": json.dumps(metrics, sort_keys=True, ensure_ascii=False),
    }
    return result, page_outputs


document_results = []
page_results = []
for selected_row in selected.toLocalIterator():
    document_result, document_pages = process_document(selected_row)
    document_results.append(document_result)
    page_results.extend(document_pages)
    print(json.dumps({
        "EVENT_ID": document_result["EVENT_ID"],
        "parse_status": document_result["parse_status"],
        "pages": document_result["pages_processed"],
        "lines": document_result["line_count"],
        "text_length": document_result["text_length"],
    }, default=str))

del engine
gc.collect()

results = spark.createDataFrame(document_results, schema=result_schema)
pages = spark.createDataFrame(page_results, schema=page_schema) if page_results else None
processed_count = len(document_results)


In [0]:
if MODE == "CANARY":
    canary_id = uuid.uuid4().hex
    canary_frame = results.withColumn("canary_id", F.lit(canary_id)).withColumn(
        "output_ts", F.current_timestamp()
    )
    target_fields = spark.table(OCR_CANARY_OUTPUT).schema.fields
    canary_aligned = canary_frame.select(*[
        F.col(field.name).cast(field.dataType).alias(field.name)
        if field.name in canary_frame.columns
        else F.lit(None).cast(field.dataType).alias(field.name)
        for field in target_fields
    ])
    canary_aligned.write.mode("append").insertInto(OCR_CANARY_OUTPUT)
    summary = {
        row["parse_status"]: int(row["count"])
        for row in results.groupBy("parse_status").count().collect()
    }
    payload = {
        "status": "canary_complete",
        "canary_id": canary_id,
        "selected": selected_count,
        "processed": processed_count,
        "states": summary,
        "target": OCR_CANARY_OUTPUT,
    }
    dbutils.jobs.taskValues.set(key="OCR_RESULT", value=json.dumps(payload))
    display(canary_aligned.select(
        "EVENT_ID", "volume_path", "parse_status", "pages_processed", "line_count",
        "mean_confidence", "text_length", "text",
    ))
    dbutils.notebook.exit(json.dumps(payload, default=str))


In [0]:
output_frame = results.withColumn("output_ts", F.current_timestamp())
(
    DeltaTable.forName(spark, OCR_OUTPUT)
    .alias("t")
    .merge(
        output_frame.alias("s"),
        "t.EVENT_ID=s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT "
        "AND t.raw_sha256=s.raw_sha256",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

if pages is not None:
    page_frame = pages.withColumn("output_ts", F.current_timestamp())
    (
        DeltaTable.forName(spark, OCR_PAGE_OUTPUT)
        .alias("t")
        .merge(
            page_frame.alias("s"),
            "t.EVENT_ID=s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT "
            "AND t.raw_sha256=s.raw_sha256 AND t.page_index=s.page_index",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

successful = results.filter(
    F.col("parse_status").isin("ocr_complete", "ocr_partial") & F.col("text").isNotNull()
)
current_target_keys = (
    spark.table(BRONZE)
    .filter(F.col("STATUS").isin(*SUPPORTED_SOURCE_STATUSES))
    .filter(F.col("BLOB_TEXT").isNull() | (F.length(F.trim(F.col("BLOB_TEXT"))) == 0))
    .select("EVENT_ID", "ADC_UPDT", "raw_sha256")
    .join(
        successful.select("EVENT_ID", "ADC_UPDT", "raw_sha256"),
        ["EVENT_ID", "ADC_UPDT", "raw_sha256"],
        "inner",
    )
    .distinct()
)
applicable = successful.join(
    current_target_keys,
    ["EVENT_ID", "ADC_UPDT", "raw_sha256"],
    "left_semi",
)

(
    DeltaTable.forName(spark, BRONZE)
    .alias("t")
    .merge(
        applicable.alias("s"),
        "t.EVENT_ID=s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT "
        "AND t.raw_sha256=s.raw_sha256",
    )
    .whenMatchedUpdate(
        condition=(
            "t.STATUS IN ('OCR queued','PDF extraction failed') "
            "AND (t.BLOB_TEXT IS NULL OR trim(t.BLOB_TEXT) = '')"
        ),
        set={
            "BLOB_TEXT": "s.text",
            "TEXT_LENGTH": "s.text_length",
            "STATUS": "'Decoded'",
            "ENCODING": "'utf-8'",
            "anon_text": "NULL",
            "parser_version": f"greatest(coalesce(t.parser_version, 0), {OCR_PARSER_VERSION})",
            "post_processor_version": (
                f"greatest(coalesce(t.post_processor_version, 0), {OCR_POST_PROCESSOR_VERSION})"
            ),
        },
    )
    .execute()
)

history_source = (
    applicable.select(
        "run_id", "EVENT_ID", "ADC_UPDT", "raw_sha256",
        F.lit(3).cast("int").alias("decompressor_version"),
        F.lit(OCR_PARSER_VERSION).cast("int").alias("parser_version"),
        F.lit(OCR_POST_PROCESSOR_VERSION).cast("int").alias("post_processor_version"),
        "status",
        F.col("truncated").alias("truncation_flag"),
        F.when(F.col("truncated"), F.lit("ocr_page_or_time_budget"))
         .otherwise(F.lit(None).cast("string")).alias("truncation_reason"),
        F.lit(None).cast("string").alias("ftfy_explain"),
        F.lit("file_volume_paddleocr_gpu").alias("decompression_strategy"),
        F.lit("ocr_paddle").alias("source"),
        F.current_date().alias("extracted_date"),
    )
)
history_aligned = history_source.select(*[
    F.col(field.name).cast(field.dataType).alias(field.name)
    if field.name in history_source.columns
    else F.lit(None).cast(field.dataType).alias(field.name)
    for field in spark.table(HISTORY).schema.fields
])
(
    DeltaTable.forName(spark, HISTORY)
    .alias("t")
    .merge(
        history_aligned.alias("s"),
        "t.run_id=s.run_id AND t.EVENT_ID=s.EVENT_ID "
        "AND t.ADC_UPDT <=> s.ADC_UPDT AND t.source='ocr_paddle'",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


In [0]:
matched_keys = current_target_keys.withColumn("_target_current", F.lit(True))
queue_updates = (
    results.join(
        matched_keys,
        ["EVENT_ID", "ADC_UPDT", "raw_sha256"],
        "left",
    )
    .withColumn(
        "next_status",
        F.when(
            F.col("parse_status") == "ocr_complete",
            F.when(F.col("_target_current"), F.lit("complete")).otherwise(F.lit("superseded")),
        )
        .when(
            F.col("parse_status") == "ocr_partial",
            F.when(F.col("_target_current"), F.lit("partial")).otherwise(F.lit("superseded")),
        )
        .when(F.col("parse_status") == "ocr_empty", F.lit("empty"))
        .when(F.col("attempt_count") >= MAX_ATTEMPTS, F.lit("failed_terminal"))
        .otherwise(F.lit("retryable")),
    )
    .withColumn(
        "last_error",
        F.when(
            F.col("next_status").isin("complete", "partial", "empty", "superseded"),
            F.lit(None).cast("string"),
        ).otherwise(F.concat_ws(": ", "status", "metrics")),
    )
)
(
    DeltaTable.forName(spark, OCR_QUEUE)
    .alias("t")
    .merge(
        queue_updates.alias("s"),
        "t.EVENT_ID=s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT "
        "AND t.raw_sha256=s.raw_sha256",
    )
    .whenMatchedUpdate(
        set={
            "status": "s.next_status",
            "lease_owner": "NULL",
            "lease_expires_ts": "NULL",
            "last_error": "s.last_error",
            "updated_ts": "current_timestamp()",
            "completed_ts": (
                "CASE WHEN s.next_status IN "
                "('complete','partial','empty','superseded','failed_terminal') "
                "THEN current_timestamp() ELSE NULL END"
            ),
        }
    )
    .execute()
)

summary = {
    row["next_status"]: int(row["count"])
    for row in queue_updates.groupBy("next_status").count().collect()
}
payload = {
    "status": "success",
    "mode": MODE,
    "selected": selected_count,
    "processed": processed_count,
    "states": summary,
    "engine": report,
}
print(json.dumps(payload, indent=2, default=str))
dbutils.jobs.taskValues.set(key="OCR_RESULT", value=json.dumps(payload, default=str))
dbutils.notebook.exit(json.dumps(payload, default=str))